# rackmains / ups / battery

**Intent:** Output the shutdown guard's required thresholds, in seconds (`rackmains_guard_{shed_max,halt_max,mains_stable,halt_delay}_seconds`), as constants from its first release, through the project path every later model uses. The signature also carries optional outputs that the default model leaves unset and the fitted model will fill: the lead times (`rackmains_guard_{shed_lead,halt_lead}_seconds`), the battery remaining (`rackmains_battery_remaining_{pct,seconds}`) and the model's energy, history, recalibration and state (`rackmains_model_{energy_used_wh,history_known,recalibrate,state_out}`).

**Notes:**
- Purpose: Give the shutdown guard conservative thresholds from its first release, through the same project path every later model uses
- Flavour: `static` now, `wasm` for the fitted model
- Data splits: `golden` the fixed samples and expected outputs; `training`, `validation` and `backtest` follow with the fit

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, version, dataset names and the constants (`model.md` 6.6).

In [ ]:
from workbench.data import golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "rackmains/ups/battery"
MODEL_NAME = "rackmains_ups_battery"
TARGETS = [{"kind": "embedded", "language": "rust"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
GOLDEN = "ups_battery_golden"
CONSTANTS = {
    "rackmains_guard_shed_max_seconds": 120,
    "rackmains_guard_halt_max_seconds": 450,
    "rackmains_guard_mains_stable_seconds": 300,
    "rackmains_guard_halt_delay_seconds": 120,
}

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `golden` rows once: a first call with no state, a row mid-drain, a row with the replace flag raised, and a request with a required input missing, which must give a signature error (`model.md` 8).

In [ ]:
def golden_rows():
    clock = "rackmains_clock_epoch_seconds"
    util = "rackmains_output_utilisation_pct"
    battery = "rackmains_ups_on_battery"
    replace = "rackmains_battery_replace"
    state = "rackmains_model_state_in"
    return {
        "case": ["first call with no state", "mid-drain with state", "replace flag raised", "required input missing"],
        "expect": ["ok", "ok", "ok", "error"],
        clock: [1793355300.0, 1793355330.0, 1793355360.0, 1793355390.0],
        util: [13, 13, 12, None],
        battery: [1, 1, 0, 1],
        replace: [0, 0, 1, 0],
        state: [None, "0.0 1235.0 0.0 1.0 1.0 1793355329.0", "0.0 0.0 600.0 0.0 1.0 1793355359.0", None],
    }


golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_rows)
golden_pl

## Package

- **Stage:** 5, package.
- **Intent:** build the signature and `golden.json`, write the `static` project and verify it. rackmains embeds it once released (R1, R3, R8).

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance("manual", "none, the rollout A starting values", [golden_path], "a conservative starting set, not fitted"),
    constants=CONSTANTS,
    sequence="baseline calls",
)
print(manifest["name"], manifest["flavour"], "written to", project_dir)

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version, "with alias preview, set release by hand after review")